## Project: DNS Tunneling Detection Machine Learning Pipeline (Stages 1 – 6)

**Stages:** Stage 1, Stage 2, Stage 3, Stage 4, Stage 5, Stage 6  

### Architectural Flow:
$$\text{Raw DNS Data} \longrightarrow \text{Imputation \& Cleaning} \longrightarrow \text{IQR Outlier Treatment} \longrightarrow \text{One-Hot Encoding} \longrightarrow \text{70/30 Stratified Split} \longrightarrow \text{Standard Scaling} \longrightarrow \text{SMOTE Balancing (Train Only)} \longrightarrow \text{EDA Visualizations} \longrightarrow \text{Export Deliverables}$$

### Objectives:
1. **Stage 1 & 2: Problem Definition & Data Acquisition**: Document the DNS tunneling cyberattack vector, source justification (CIC benchmark dataset), licensing, records, and feature schema.
2. **Stage 3 & 4: Data Preparation & Preprocessing**:
   - Address packet dropouts via Median Imputation (numerical) and Mode Imputation (categorical).
   - Treat capture artifacts and length glitches via Interquartile Range (IQR).
   - Encode nominal DNS record types (`A`, `TXT`, `CNAME`, `PTR`) via One-Hot Encoding (with theoretical justification over Label Encoding).
   - Apply `StandardScaler` to equalize feature scales and prevent `query_length` from dominating `domain_entropy` in distance-based algorithms.
3. **Stage 5: Data Balancing (Crucial Step)**:
   - Handle the severe real-world cybersecurity imbalance (99% Benign vs 1% Malicious).
   - **MANDATORY ANTI-LEAKAGE RULE**: Enforce `train_test_split` (70/30 stratified) **BEFORE** applying SMOTE.
   - Synthesize minority samples using $k$-NN ($k=5$) on training set only.
4. **Stage 6: Exploratory Data Analysis (EDA)**:
   - Univariate KDE/histograms of `query_length`.
   - Bivariate box/violin plots of `domain_entropy`.
   - Feature correlation heatmap identifying collinear features ($|r| > 0.90$) for Harsh's feature selection stage.
5. **Deliverables Export**:
   - Cleaned, scaled, balanced `X_train_bal`, `X_test`, `y_train_bal`, `y_test`.
   - High-resolution publication-grade EDA figures and fitted pipeline artifacts.


In [ ]:
import ast
import os
import sys
import zipfile
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Visual styling
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Project paths
DATA_DIR = Path('.')
FIGURES_DIR = Path('figures')
PROCESSED_DIR = Path('data/processed')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment and directories configured successfully.")


---
## Stage 1 & 2: Problem Definition & Data Acquisition

### 1. The Threat Model: DNS Tunneling Exfiltration
* **Mechanism**: Domain Name System (DNS) protocol operates over UDP/TCP port 53 and is fundamentally trusted in enterprise network environments to resolve hostnames to IP addresses. Firewalls, egress filters, and proxies rarely block or inspect recursive DNS lookups.
* **The Exploit**: Cyber adversaries weaponize this trust to bypass firewalls and Data Loss Prevention (DLP) systems:
  1. Attackers encode stolen data (passwords, encryption keys, confidential documents) into the labels of a Fully Qualified Domain Name (FQDN), e.g., `chunk1_pwd123.evil-domain.com`.
  2. The local DNS resolver forwards the query to the attacker's Authoritative Name Server.
  3. The attacker's server logs the query, decodes the subdomain payload, and optionally returns instructions or payloads hidden inside `TXT`, `NULL`, or `CNAME` resource records.
* **Why Traditional Defenses Fail**: Static signature matching and domain blacklists fail against modern Domain Generation Algorithms (DGA) and fast-flux networks. Anomaly detection via Machine Learning on DNS statistical features is essential.

### 2. Dataset Documentation & Justification
* **Dataset**: **CIC-Bell-DNS-EXF-2021 / CIRA-CIC-DoHBrw-2020** (Canadian Institute for Cybersecurity, University of New Brunswick).
* **Licensing**: Open academic research dataset (Creative Commons / CIC Academic Use License).
* **Scenarios Captured**:
  - **12 Attack Scenarios**: 6 distinct exfiltration data types (`audio`, `compressed`, `exe`, `image`, `text`, `video`) across 2 transmission intensities (`heavy` and `light`).
  - **Benign Baseline**: Normal enterprise user browsing traffic, routine OS updates, background cloud telemetry.
* **Feature Representations**:
  - **Stateful Features**: Aggregated statistical distributions across DNS response windows (Resource Record types, entropy, TTL statistics, autonomous system numbers, record frequencies).
  - **Stateless Features**: Per-query lexical features (FQDN length, subdomain character distributions, Shannon entropy).
* **Pipeline Focus**: We ingest the stateful feature representations as they contain DNS resource record types (`rr_type`), query length (`rr_name_length`), and domain entropy (`rr_name_entropy`), perfectly aligning with Krishna's Blueprint.


In [ ]:
def load_dns_dataset(base_dir=Path('.')):
    """
    Ingests raw stateful DNS records from compressed archives:
    - Benign.zip (root baseline)
    - Attack_light (Benign.zip, Attacks.zip)
    - Attack_heavy (Benign.zip, Attacks.zip)
    """
    sources = [
        (base_dir / 'Benign.zip', 0, 'root_benign'),
        (base_dir / 'Attack_light' / 'Benign.zip', 0, 'light_benign'),
        (base_dir / 'Attack_light' / 'Attacks.zip', 1, 'light_attack'),
        (base_dir / 'Attack_heavy' / 'Benign.zip', 0, 'heavy_benign'),
        (base_dir / 'Attack_heavy' / 'Attacks.zip', 1, 'heavy_attack'),
    ]
    
    records = []
    loaded_files = 0
    
    for zip_path, label, source_category in sources:
        if not zip_path.exists():
            print(f"Warning: {zip_path} not found. Skipping.")
            continue
        with zipfile.ZipFile(zip_path, 'r') as z:
            for fname in z.namelist():
                if fname.lower().endswith('.csv') and 'stateful' in fname.lower() and '__macosx' not in fname.lower():
                    with z.open(fname) as f:
                        df_part = pd.read_csv(f, low_memory=False)
                        df_part['label'] = label
                        df_part['source_category'] = source_category
                        df_part['source_file'] = fname
                        records.append(df_part)
                        loaded_files += 1
                        
    if not records:
        raise FileNotFoundError("No stateful DNS CSV files were found in specified archives.")
        
    df_raw = pd.concat(records, ignore_index=True)
    print(f"Successfully loaded {loaded_files} stateful CSV files.")
    print(f"Total raw records: {len(df_raw):,}")
    print(f"Total columns: {df_raw.shape[1]}")
    print(f"Class breakdown:
{df_raw['label'].value_counts().rename({0: 'Benign (0)', 1: 'Malicious (1)'})}")
    return df_raw

df_raw = load_dns_dataset(DATA_DIR)
df_raw.head(3)


---
## Stage 3 & 4: Data Preparation & Preprocessing

### 1. Packet Loss and Missing Value Imputation
* **The Network Phenomenon**: High-speed network capture appliances (such as `pcap` probes or TAP devices) frequently drop frames during high traffic volume bursts. When an aggregation window closes without observing expected response packets, null (`NaN`) values appear in the dataset.
* **Missing Value Handling Strategy**:
  1. **Threshold Assessment**: If missing values in any record account for $< 1\%$ of the dataset, dropping them is permissible, but systematic imputation preserves sample size.
  2. **Median Imputation for Numerical Features**:
     $$\text{Median}(X) = X_{\left(\frac{n+1}{2}\right)}$$
     *Why Median over Mean?* Network traffic metrics (e.g., query lengths, TTL variances, packet counts) exhibit heavy positive skewness with long tails. Mean imputation is heavily distorted by extreme values, whereas the median represents the true typical central tendency without outlier vulnerability.
  3. **Mode Imputation for Categorical Features**:
     $$\text{Mode}(X) = \arg\max_c \left(\sum_{i=1}^n \mathbb{I}(X_i = c)\right)$$
     *Why Mode?* Record types are categorical. Mode assigns the most prevalent genuine record type (`A` or `PTR`) without generating fractional or non-existent categories.
  4. **Container Parsing**: Columns serialized as Python literal strings (e.g. `set()`, `{}`, `[1, 1]`) are parsed into numeric cardinalities (unique element counts).


In [ ]:
def parse_container_cardinality(val):
    """Converts string representation of containers to integer cardinality count."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s in ('set()', '{}', '[]', '()', 'None', ''):
        return 0
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (set, frozenset, list, tuple)):
            return len(set(obj))
        elif isinstance(obj, dict):
            return len(obj)
    except Exception:
        pass
    return 1

def clean_record_type(val):
    """Extracts clean primary DNS record type name (A, TXT, CNAME, PTR, AAAA)."""
    if pd.isna(val):
        return 'UNKNOWN'
    s = str(val).strip()
    if s in ('set()', '{}', '[]', '{None}', 'None', ''):
        return 'NONE'
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, set):
            items = [str(x) for x in obj if x is not None]
            if items:
                return sorted(items)[0] # Primary record type
            return 'NONE'
    except Exception:
        pass
    cleaned = s.replace("{'", "").replace("'}", "").replace("{", "").replace("}", "").strip()
    return cleaned if cleaned else 'UNKNOWN'

# Create working copy and standardize canonical aliases
df_clean = df_raw.copy()

# De-duplicate exact identical rows
initial_rows = len(df_clean)
df_clean.drop_duplicates(subset=[c for c in df_clean.columns if c not in ['source_file', 'source_category']], inplace=True)
print(f"Removed {initial_rows - len(df_clean):,} exact duplicate rows. Retained {len(df_clean):,} rows.")

# Parse container columns to numeric cardinality
container_cols = ['distinct_ip', 'unique_country', 'unique_asn', 'distinct_domains', 'unique_ttl']
for col in container_cols:
    if col in df_clean.columns:
        df_clean[col + '_count'] = df_clean[col].apply(parse_container_cardinality)

# Clean categorical record type
df_clean['rr_type_clean'] = df_clean['rr_type'].apply(clean_record_type)

# Define explicit domain feature aliases matching the blueprint
df_clean['query_length'] = pd.to_numeric(df_clean['rr_name_length'], errors='coerce')
df_clean['domain_entropy'] = pd.to_numeric(df_clean['rr_name_entropy'], errors='coerce')

# Parse binary reverse_dns indicator
if 'reverse_dns' in df_clean.columns:
    df_clean['reverse_dns_known'] = (df_clean['reverse_dns'].astype(str).str.strip().str.lower() != 'unknown').astype(int)

# Check missing value statistics
print("\n--- Missing Value Report ---")
null_counts = df_clean.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "Zero missing values detected across all core features.")


### 2. Outlier Treatment: Interquartile Range (IQR) Method
* **DNS Capture Errors**: Packet capture buffers occasionally log corrupt field lengths (e.g., query lengths recorded as $9999$ due to buffer overflows or framing errors).
* **The IQR Mathematical Formulation**:
  Let $Q_1$ be the 25th percentile and $Q_3$ be the 75th percentile of feature $X$:
  $$\text{IQR} = Q_3 - Q_1$$
  $$\text{Lower Bound} = \max\left(0, Q_1 - 1.5 \times \text{IQR}\right)$$
  $$\text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$
* **Preserving Valid Cyber Attack Signatures**:
  Unlike standard sensors where any outlier is noise, **DNS Tunneling relies on abnormally long subdomains** to maximize exfiltration throughput!
  Therefore, we apply the IQR threshold to cap impossible capture corruptions (e.g., lengths exceeding RFC 1035 limits where maximum DNS query name length is 255 characters, or corrupt values $> 9000$) while preserving legitimate long tunneling queries.


In [ ]:
def treat_outliers_iqr(df, feature_col, max_cap=255.0):
    """
    Calculates Q1, Q3, and IQR to detect anomalies and apply domain-informed upper capping.
    RFC 1035 sets absolute maximum DNS query length to 255 characters.
    """
    Q1 = df[feature_col].quantile(0.25)
    Q3 = df[feature_col].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = max(0.0, Q1 - 1.5 * IQR)
    upper_limit = min(max_cap, Q3 + 3.0 * IQR) # Preserves heavy tunneling tails up to protocol maximum
    
    anomalies = df[(df[feature_col] > max_cap) | (df[feature_col] < 0)]
    print(f"Feature '{feature_col}': Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
    print(f"Operational Bounds: [{lower_limit:.2f}, {upper_limit:.2f}] (Hard protocol cap: {max_cap})")
    print(f"Corrupt packet anomalies detected (> {max_cap}): {len(anomalies)}")
    
    # Clip extreme corruption to valid domain limits
    df[feature_col] = df[feature_col].clip(lower=lower_limit, upper=upper_limit)
    return df

df_clean = treat_outliers_iqr(df_clean, 'query_length', max_cap=255.0)
df_clean = treat_outliers_iqr(df_clean, 'domain_entropy', max_cap=8.0) # Theoretical Shannon entropy limit for DNS chars


---
## Stage 4: Categorical Encoding & Feature Scaling

### 1. Categorical Encoding: One-Hot Encoding (OHE)
* **The Feature**: DNS Resource Record types (`rr_type_clean`: `A`, `TXT`, `CNAME`, `PTR`, `AAAA`, etc.).
* **Method**: One-Hot Encoding via `pd.get_dummies`.
* **Theoretical Justification (Why NOT Label Encoding?)**:
  - DNS record types are **nominal categorical variables**. There is no intrinsic mathematical hierarchy or ordering between an `A` record ($32$-bit IPv4 address mapping), a `TXT` record (arbitrary text payload carrier), and a `CNAME` record (canonical domain alias).
  - If we applied **Label Encoding** ($A = 1, TXT = 2, CNAME = 3, PTR = 4$), an algorithm would interpret:
    $$|PTR - A| = 3 > |TXT - A| = 1$$
    This falsely assumes a `PTR` record is "thrice as distant" from an `A` record as a `TXT` record is.
  - In distance-based models (such as Jaimin's SVM and Neural Network), artificial ordinality distorts the hyperplanes and gradient descents.
  - **One-Hot Encoding** represents each category as an orthogonal binary unit basis vector in $\mathbb{R}^k$, guaranteeing that all categories are equidistant:
    $$\|e_i - e_j\| = \sqrt{2} \quad \forall i \ne j$$


In [ ]:
# Identify top record type categories and One-Hot Encode
print("DNS Record Types distribution:")
print(df_clean['rr_type_clean'].value_counts())

# Generate One-Hot Encoded columns
ohe_dummies = pd.get_dummies(df_clean['rr_type_clean'], prefix='rr_type', dtype=float)
print(f"\nGenerated {ohe_dummies.shape[1]} One-Hot Encoded binary features:")
print(list(ohe_dummies.columns))

# Join dummies back to cleaned dataset
df_encoded = pd.concat([df_clean, ohe_dummies], axis=1)


---
## Stage 5: Data Balancing

### 1. The Cybersecurity Imbalance Reality
* In real-world enterprise network operations, $> 99\%$ of DNS queries are legitimate routine traffic ($	ext{Benign} = 0$), while stealthy tunneling attacks comprise $< 1\%$ ($	ext{Malicious} = 1$).
* **The Accuracy Paradox**: A naive majority-class classifier that blindly classifies every query as "Benign" achieves $99\%$ accuracy while having **$0\%$ Recall on cyber attacks**, rendering the security operations center completely blind.

### 2. MANDATORY GUIDELINE RULE: `train_test_split` BEFORE SMOTE!
* **The Golden Rule of ML Hygiene**:
  $$\text{Splitting (70/30 Stratified)} \implies \text{Fitting Scaler on Train} \implies \text{SMOTE on Train ONLY}$$
* **Why Data Leakage Occurs if SMOTE is Applied First**:
  1. SMOTE synthesizes new minority points by interpolating between neighboring instances in feature space:
     $$x_{\text{synthetic}} = x_i + \lambda (x_{(k)} - x_i), \quad \lambda \in [0, 1]$$
  2. If SMOTE is applied across the entire dataset before splitting, a synthetic point derived from a future test sample will end up inside the training set.
  3. The model would effectively "memorize" properties of the test set during training.
  4. This produces **artificially inflated evaluation metrics** that catastrophically fail in real-world deployment.
* **The Proper Protocol**:
  - We execute a **70/30 Stratified Split** first.
  - The test set $(X_{\text{test}}, y_{\text{test}})$ is sealed and untouched, maintaining the authentic operational class distribution.
  - SMOTE is applied **strictly** to $(X_{\text{train}}, y_{\text{train}})$.
  - `StandardScaler` is fitted **strictly** on $X_{\text{train}}$ and applied to transform both $X_{\text{train}}$ and $X_{\text{test}}$.


In [ ]:
# Feature Selection for Modeling Matrix
candidate_features = [
    # Primary domain features
    'query_length', 'domain_entropy',
    # Record frequency features
    'A_frequency', 'NS_frequency', 'CNAME_frequency', 'SOA_frequency', 'NULL_frequency',
    'PTR_frequency', 'HINFO_frequency', 'MX_frequency', 'TXT_frequency', 'AAAA_frequency',
    'SRV_frequency', 'OPT_frequency',
    # Resource record counts and statistics
    'rr_count', 'a_records', 'ttl_mean', 'ttl_variance',
    # Container counts
    'distinct_ip_count', 'unique_country_count', 'unique_asn_count', 'distinct_domains_count', 'unique_ttl_count',
    'reverse_dns_known'
] + list(ohe_dummies.columns)

# Verify presence and convert to numeric float matrix
feature_cols = [c for c in candidate_features if c in df_encoded.columns]
X = df_encoded[feature_cols].astype(float).fillna(0.0)
y = df_encoded['label'].astype(int)

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")

# MANDATORY RULE: Stratified 70/30 Split BEFORE SMOTE
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"\n--- Split Distribution (70% Train, 30% Test) ---")
print(f"X_train rows: {len(X_train_raw):,} | Benign: {(y_train == 0).sum():,} | Attack: {(y_train == 1).sum():,} ({y_train.mean():.2%})")
print(f"X_test rows:  {len(X_test_raw):,} | Benign: {(y_test == 0).sum():,} | Attack: {(y_test == 1).sum():,} ({y_test.mean():.2%})")


### 2. Feature Scaling: StandardScaler ($z$-Score Normalization)
* **Mathematical Formula**:
  $$z = \frac{x - \mu}{\sigma}$$
* **Theoretical Justification**:
  - `query_length` spans values up to $255$, whereas `domain_entropy` resides strictly in the $[0.5, 4.5]$ interval.
  - In distance-based models such as **Jaimin's Support Vector Machine (SVM with RBF Kernel: $\exp(-\gamma \|x_i - x_j\|^2)$)** and **Neural Networks (weight gradient updates $\nabla_w L \propto x$)**, Euclidean distance calculation is dominated by `query_length` ($255^2 = 65,025$) over `domain_entropy` ($4^2 = 16$).
  - Without scaling, the model treats `domain_entropy` as negligible noise.
  - Furthermore, **SMOTE uses $k$-NN Euclidean distance** to generate synthetic minority samples. Unscaled features cause SMOTE to interpolate exclusively along `query_length`, synthesizing physically distorted network packets.
  - `StandardScaler` fitted strictly on $X_{\text{train}}$ gives all features zero mean and unit variance.


In [ ]:
# Fit StandardScaler strictly on Training Data
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=X_train_raw.columns,
    index=X_train_raw.index
)

# Transform Test Data using train parameters
X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index
)

print("StandardScaler successfully fitted on X_train and applied to X_train and X_test.")
print(f"X_train mean across features: {X_train_scaled.mean().mean():.4f} (std: {X_train_scaled.std().mean():.4f})")


### 3. Applying SMOTE (Synthetic Minority Over-sampling Technique)
* **Mechanism**: Rather than simple duplication (which causes severe decision boundary overfitting), SMOTE takes each minority sample, finds its $k$-nearest neighbors ($k=5$), and creates synthetic points along the connecting line segments:
  $$x_{\text{new}} = x_i + u \cdot (x_{\text{neighbor}} - x_i), \quad u \sim U(0, 1)$$
* We apply SMOTE **strictly to $(X_{\text{train\_scaled}}, y_{\text{train}})$**.


In [ ]:
# Apply SMOTE to Training set ONLY
smote = SMOTE(sampling_strategy='auto', k_neighbors=5, random_state=RANDOM_STATE)
X_train_bal_arr, y_train_bal_arr = smote.fit_resample(X_train_scaled, y_train)

X_train_bal = pd.DataFrame(X_train_bal_arr, columns=X_train_scaled.columns)
y_train_bal = pd.Series(y_train_bal_arr, name='label')

print("--- SMOTE Balancing Results ---")
print(f"X_train BEFORE SMOTE: {len(X_train_scaled):,} rows | Distribution: {dict(y_train.value_counts())}")
print(f"X_train AFTER SMOTE:  {len(X_train_bal):,} rows | Distribution: {dict(y_train_bal.value_counts())}")
print(f"X_test (PRISTINE):    {len(X_test):,} rows | Distribution: {dict(y_test.value_counts())}")


---
## Stage 6: Exploratory Data Analysis (EDA)

Krishna's EDA is **decision-driven**, designed to yield direct empirical findings that guide **Harsh's Feature Selection** and **Jaimin's Model Architecture**:
1. **Univariate Analysis (`query_length`)**: Demonstrates benign traffic clustering around $15-20$ characters vs the heavy spread of malicious exfiltration.
2. **Bivariate Analysis (`domain_entropy`)**: Illustrates high Shannon entropy in tunneling traffic due to Base32/Base64/encrypted payload strings.
3. **Correlation Matrix Heatmap**: Identifies severe multicollinearity ($|r| > 0.90$) to instruct Harsh on which redundant features to drop.
4. **Balancing Comparison**: Quantifies class representation before and after SMOTE.


In [ ]:
# -------------------------------------------------------------------------
# EDA Chart 1: Univariate Analysis - query_length Histogram & KDE
# -------------------------------------------------------------------------
plt.figure(figsize=(10, 5))
benign_lengths = df_clean[df_clean['label'] == 0]['query_length']
attack_lengths = df_clean[df_clean['label'] == 1]['query_length']

sns.histplot(benign_lengths, color='#1f77b4', label=f'Benign (Mean={benign_lengths.mean():.1f})',
             kde=True, stat="density", bins=30, alpha=0.5)
sns.histplot(attack_lengths, color='#d62728', label=f'Malicious Attack (Mean={attack_lengths.mean():.1f})',
             kde=True, stat="density", bins=30, alpha=0.5)

plt.axvline(x=20, color='black', linestyle='--', alpha=0.7, label='Standard Benign Ceiling (~20 chars)')
plt.title("EDA 1: Univariate Analysis - DNS Query Length Distribution", fontsize=14, fontweight='bold')
plt.xlabel("Query Length (Characters)", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.xlim(0, 50)
plt.legend(frameon=True, facecolor='white', loc='upper right')
plt.tight_layout()

chart1_path = FIGURES_DIR / 'eda_1_univariate_query_length.png'
plt.savefig(chart1_path, dpi=300)
plt.show()
print(f"Saved: {chart1_path}")


In [ ]:
# -------------------------------------------------------------------------
# EDA Chart 2: Bivariate Analysis - domain_entropy Boxplot & Violinplot
# -------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#1f77b4', '#d62728']
labels = ['Benign (0)', 'Malicious Tunneling (1)']

# Violin plot
sns.violinplot(ax=axes[0], data=df_clean, x='label', y='domain_entropy', hue='label', palette=colors, legend=False, inner='quartile')
axes[0].set_title("Domain Entropy Density by Class", fontsize=13, fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(labels)
axes[0].set_xlabel("Traffic Class")
axes[0].set_ylabel("Shannon Entropy (bits)")

# Box plot
sns.boxplot(ax=axes[1], data=df_clean, x='label', y='domain_entropy', hue='label', palette=colors, legend=False, width=0.4)
axes[1].set_title("Domain Entropy Quartiles & Outliers", fontsize=13, fontweight='bold')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(labels)
axes[1].set_xlabel("Traffic Class")
axes[1].set_ylabel("Shannon Entropy (bits)")

plt.suptitle("EDA 2: Bivariate Analysis - Domain Entropy Across Classes", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

chart2_path = FIGURES_DIR / 'eda_2_bivariate_domain_entropy.png'
plt.savefig(chart2_path, dpi=300)
plt.show()
print(f"Saved: {chart2_path}")


In [ ]:
# -------------------------------------------------------------------------
# EDA Chart 3: Correlation Matrix Heatmap & Multicollinearity Analysis
# -------------------------------------------------------------------------
corr_features = [
    'query_length', 'domain_entropy', 'A_frequency', 'NS_frequency',
    'CNAME_frequency', 'PTR_frequency', 'TXT_frequency', 'rr_count',
    'a_records', 'ttl_mean', 'ttl_variance', 'distinct_ip_count',
    'unique_country_count', 'unique_asn_count', 'distinct_domains_count'
]
corr_features = [c for c in corr_features if c in X_train_raw.columns]

corr_matrix = X_train_raw[corr_features].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation Coefficient'}
)
plt.title("EDA 3: Feature Correlation Matrix Heatmap", fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

chart3_path = FIGURES_DIR / 'eda_3_correlation_matrix.png'
plt.savefig(chart3_path, dpi=300)
plt.show()
print(f"Saved: {chart3_path}")

# Detect highly collinear pairs (|r| > 0.90) for Harsh's Feature Selection
print("\n=== Actionable Recommendations for Harsh (Feature Selection) ===")
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        col1, col2 = corr_matrix.columns[i], corr_matrix.columns[j]
        r_val = corr_matrix.iloc[i, j]
        if abs(r_val) >= 0.90:
            high_corr_pairs.append((col1, col2, r_val))

if high_corr_pairs:
    print("ALERT: The following feature pairs have extreme correlation (|r| >= 0.90) creating multicollinearity:")
    for c1, c2, r in high_corr_pairs:
        print(f"  - '{c1}' <-> '{c2}': r = {r:.3f} --> Recommend Harsh drop '{c2}' to reduce dimensionality and variance.")
else:
    print("No severe multicollinear pairs (|r| >= 0.90) identified among primary continuous features.")


In [ ]:
# -------------------------------------------------------------------------
# EDA Chart 4: Class Distribution Comparison (Raw vs Test vs Post-SMOTE)
# -------------------------------------------------------------------------
plt.figure(figsize=(9, 5))

stages = ['Raw Dataset', 'Test Split (30%)', 'Train Pre-SMOTE (70%)', 'Train Post-SMOTE']
benign_counts = [
    (df_clean['label'] == 0).sum(),
    (y_test == 0).sum(),
    (y_train == 0).sum(),
    (y_train_bal == 0).sum()
]
attack_counts = [
    (df_clean['label'] == 1).sum(),
    (y_test == 1).sum(),
    (y_train == 1).sum(),
    (y_train_bal == 1).sum()
]

x = np.arange(len(stages))
width = 0.35

plt.bar(x - width/2, benign_counts, width, label='Benign (0)', color='#1f77b4')
plt.bar(x + width/2, attack_counts, width, label='Attack (1)', color='#d62728')

for i in range(len(stages)):
    plt.text(i - width/2, benign_counts[i] + 1500, f"{benign_counts[i]:,}", ha='center', fontsize=9)
    plt.text(i + width/2, attack_counts[i] + 1500, f"{attack_counts[i]:,}", ha='center', fontsize=9)

plt.title("EDA 4: Pipeline Class Distribution Across Processing Stages", fontsize=14, fontweight='bold')
plt.ylabel("Number of Records", fontsize=12)
plt.xticks(x, stages, rotation=10)
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()

chart4_path = FIGURES_DIR / 'eda_4_class_distribution_smote.png'
plt.savefig(chart4_path, dpi=300)
plt.show()
print(f"Saved: {chart4_path}")


---
## Deliverables Export & Handoff Pack

We now export the finalized pipeline assets for **Harsh** and **Jaimin**:
1. `X_train_bal.csv`: SMOTE-balanced, scaled features ready for training.
2. `y_train_bal.csv`: SMOTE-balanced target labels.
3. `X_test.csv`: Pristine, scaled features representing real-world distribution.
4. `y_test.csv`: Pristine test labels.
5. `preprocessor_scaler.joblib`: Serialized `StandardScaler` pipeline for production inference.
6. Summary diagnostics table verifying zero NaNs and identical feature columns.


In [ ]:
# Save processed datasets to disk
print("Exporting finalized datasets to 'data/processed/'...")
X_train_bal.to_csv(PROCESSED_DIR / 'X_train_bal.csv', index=False)
y_train_bal.to_csv(PROCESSED_DIR / 'y_train_bal.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

# Save scaler pipeline
joblib.dump(scaler, PROCESSED_DIR / 'preprocessor_scaler.joblib')

# Final Verification Sanity Check
print("\n=================== HANDOFF INTEGRITY VERIFICATION ===================")
assert X_train_bal.shape[1] == X_test.shape[1], "Feature count mismatch between train and test!"
assert not X_train_bal.isnull().any().any(), "NaNs detected in X_train_bal!"
assert not X_test.isnull().any().any(), "NaNs detected in X_test!"
assert len(X_train_bal) == len(y_train_bal), "Train feature/target length mismatch!"
assert len(X_test) == len(y_test), "Test feature/target length mismatch!"

print("  STATUS: ALL SANITY CHECKS PASSED SUCCESSFULLY!")
print(f"  X_train_bal shape: {X_train_bal.shape} (Balanced 50/50)")
print(f"  X_test shape:      {X_test.shape} (Natural 30% holdout)")
print(f"  Total Features:    {X_train_bal.shape[1]}")
print(f"  Artifacts Path:    {PROCESSED_DIR.resolve()}")
print(f"  Figures Path:      {FIGURES_DIR.resolve()}")
print("=======================================================================")
